In [52]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from time import time
# --------------------------------------------------
# CONSTANTES
# --------------------------------------------------
SEED     = 42
N_SPLITS = 5
GAMMAS   = np.logspace(-3, 1, 5)

# --------------------------------------------------
# 1) Definición de modelos: Regresión Logística y SVM (linear, poly, rbf)
# --------------------------------------------------
def classical_models():
    return {
        "LogReg": (
            Pipeline([
                ("scaler", StandardScaler()),
                ("clf",    LogisticRegression(max_iter=1000, random_state=SEED))
            ]),
            {"clf__C": [0.01, 0.1, 1, 10, 100]}
        ),
        "SVM-linear": (
            Pipeline([
                ("scaler", StandardScaler()),
                ("clf",    SVC(kernel="linear", probability=True, random_state=SEED))
            ]),
            {"clf__C": [0.01, 0.1, 1, 10, 100]}
        ),
        "SVM-poly": (
            Pipeline([
                ("scaler", StandardScaler()),
                ("clf",    SVC(kernel="poly", probability=True, random_state=SEED))
            ]),
            {
                "clf__degree": [2, 3, 4],
                "clf__C":      [0.1, 1, 10],
                "clf__coef0":  [0, 1]
            }
        ),
        "SVM-rbf": (
            Pipeline([
                ("scaler", StandardScaler()),
                ("clf",    SVC(kernel="rbf", probability=True, random_state=SEED))
            ]),
            {
                "clf__C":     [0.1, 1, 10],
                "clf__gamma": GAMMAS
            }
        )
    }

# --------------------------------------------------
# 2) Función de evaluación: ajusta y mide los modelos
# --------------------------------------------------
def evaluate_classical(X_tr, y_tr, X_ts, y_ts):
    results = []
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    for name, (pipeline, param_grid) in classical_models().items():
        start = time()
        if param_grid:
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring="roc_auc",
                cv=cv,
                n_jobs=-1
            )
            search.fit(X_tr, y_tr)
            model = search.best_estimator_
            best_params = search.best_params_
        else:

            model = pipeline.fit(X_tr, y_tr)
            best_params = {}

        y_pred = model.predict(X_ts)
        y_proba = model.predict_proba(X_ts)[:, 1]

        results.append({
            "model":         name,
            "accuracy_test": accuracy_score(y_ts, y_pred),
            "f1_test":       f1_score(y_ts, y_pred),
            "auc_test":      roc_auc_score(y_ts, y_proba),
            "best_params":   best_params,
            "time":  time() - start
        })

    return pd.DataFrame(results).sort_values("accuracy_test").reset_index(drop=True)

# --------------------------------------------------
# 3) Script principal
# --------------------------------------------------
def main():
    # Carga de datos
    df = pd.read_csv("Heart Prediction Quantum Dataset.csv")
    X = df.drop("HeartDisease", axis=1)
    y = df["HeartDisease"]

    # División entrenamiento/prueba
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2,
        random_state=SEED,
        stratify=y
    )

    # Evaluación de modelos
    results = evaluate_classical(X_train, y_train, X_test, y_test)
    print(results.sort_values('accuracy_test', ascending = False))

if __name__ == "__main__":
    main()


        model  accuracy_test   f1_test  auc_test  \
3     SVM-rbf           0.94  0.950820  0.982500   
2  SVM-linear           0.93  0.942149  0.983750   
1    SVM-poly           0.92  0.935484  0.979167   
0      LogReg           0.92  0.934426  0.982917   

                                         best_params      time  
3                 {'clf__C': 10, 'clf__gamma': 0.01}  3.414490  
2                                     {'clf__C': 10}  1.320297  
1  {'clf__C': 0.1, 'clf__coef0': 1, 'clf__degree'...  1.952087  
0                                      {'clf__C': 1}  0.775906  
